# Lab 4: Financial Research Agent
**Goal:** turn the Lab 3 RAG system into an *agent* that decides for itself which tool to use, then pauses for human approval on high-stakes answers.

**Why:** plain RAG always does the same thing: retrieve, then answer. An agent can choose between searching documents, doing arithmetic, or summarising a page, and can chain them. This is Module 4, sections 1, 2 and 4 (tools, function calling, ReAct) and section 8 (human-in-the-loop controls).

This notebook is self-contained: Exercise 1 rebuilds the small Lab 3 index, so you do not need to keep the Lab 3 notebook open.

## Exercise 0: Choose your model provider
**Why we do this:** agents need a model that supports *tool calling* (also called function calling): the model replies with a request like "call calculate_ratio with these numbers" instead of plain text. Not every model does this well, so the model choice matters more here than in Lab 3.

**What the code does:** defines one `chat(messages, tools)` function that works with six providers and always returns the reply in the same shape. Five providers use the OpenAI-style API (so they share code); Anthropic has its own API, so a small adapter converts the messages.

| Provider | Cost | Notes |
|---|---|---|
| `ollama` | Free, local | Run this notebook locally; Colab cannot reach your machine. Use a model tagged "tools" in the Ollama library. Small models call tools less reliably. |
| `groq` | Free tier | Fast hosted open models. Rate-limited. |
| `gemini` | Free tier | Google AI Studio key. Used here through Google's OpenAI-compatible endpoint. |
| `openrouter` | Free models with limits | One key, many models. The `openrouter/free` router picks a free model; free limits are small, and an agent makes several calls per question. |
| `openai` | Paid | Works well with tools. |
| `anthropic` | Paid | Works well with tools. |

**Important:** model names and free-tier limits change often. If a model is rejected, check the provider's current model list and edit `MODEL`. Limits count every model call, and an agent makes several per question.

In [ ]:
%pip install -q chromadb pypdf openai anthropic

In [ ]:
import os, json, getpass

PROVIDER = "groq"   # ollama | groq | gemini | openrouter | openai | anthropic

PROVIDERS = {
    "ollama":     {"base_url": "http://localhost:11434/v1",                                "key_env": None,                "model": "llama3.1:8b"},
    "groq":       {"base_url": "https://api.groq.com/openai/v1",                          "key_env": "GROQ_API_KEY",       "model": "llama-3.3-70b-versatile"},
    "gemini":     {"base_url": "https://generativelanguage.googleapis.com/v1beta/openai/", "key_env": "GEMINI_API_KEY",     "model": "gemini-2.5-flash"},
    "openrouter": {"base_url": "https://openrouter.ai/api/v1",                            "key_env": "OPENROUTER_API_KEY", "model": "openrouter/free"},
    "openai":     {"base_url": None,                                                      "key_env": "OPENAI_API_KEY",     "model": "gpt-4o-mini"},
    "anthropic":  {"base_url": None,                                                      "key_env": "ANTHROPIC_API_KEY",  "model": "claude-sonnet-5-5"},
}
cfg = PROVIDERS[PROVIDER]
MODEL = cfg["model"]
if cfg["key_env"] and not os.getenv(cfg["key_env"]):
    os.environ[cfg["key_env"]] = getpass.getpass(cfg["key_env"] + ": ")

def to_anthropic(messages):
    # Convert OpenAI-style messages into Anthropic's format.
    system = messages[0]["content"]
    out = []
    for m in messages[1:]:
        if m["role"] == "user":
            out.append({"role": "user", "content": m["content"]})
        elif m["role"] == "assistant":
            blocks = []
            if m.get("content"):
                blocks.append({"type": "text", "text": m["content"]})
            for tc in m.get("tool_calls", []):
                blocks.append({"type": "tool_use", "id": tc["id"], "name": tc["function"]["name"],
                               "input": json.loads(tc["function"]["arguments"] or "{}")})
            out.append({"role": "assistant", "content": blocks})
        elif m["role"] == "tool":
            block = {"type": "tool_result", "tool_use_id": m["tool_call_id"], "content": m["content"]}
            if out and out[-1]["role"] == "user" and isinstance(out[-1]["content"], list):
                out[-1]["content"].append(block)
            else:
                out.append({"role": "user", "content": [block]})
    return system, out

def chat(messages, tools=None):
    # Returns {"role": "assistant", "content": str, "tool_calls": [...] (optional)}
    if PROVIDER == "anthropic":
        import anthropic
        system, msgs = to_anthropic(messages)
        kw = dict(model=MODEL, max_tokens=1000, temperature=0, system=system, messages=msgs)
        if tools:
            kw["tools"] = [{"name": t["function"]["name"], "description": t["function"]["description"],
                            "input_schema": t["function"]["parameters"]} for t in tools]
        r = anthropic.Anthropic().messages.create(**kw)
        msg = {"role": "assistant", "content": "".join(b.text for b in r.content if b.type == "text")}
        calls = [{"id": b.id, "type": "function", "function": {"name": b.name, "arguments": json.dumps(b.input)}}
                 for b in r.content if b.type == "tool_use"]
        if calls:
            msg["tool_calls"] = calls
        return msg
    from openai import OpenAI
    key = os.getenv(cfg["key_env"]) if cfg["key_env"] else "ollama"
    client = OpenAI(base_url=cfg["base_url"], api_key=key) if cfg["base_url"] else OpenAI(api_key=key)
    kw = dict(model=MODEL, temperature=0, messages=messages)
    if tools:
        kw["tools"] = tools
    m = client.chat.completions.create(**kw).choices[0].message
    msg = {"role": "assistant", "content": m.content or ""}
    if m.tool_calls:
        msg["tool_calls"] = [{"id": tc.id, "type": "function",
                              "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                             for tc in m.tool_calls]
    return msg

print(chat([{"role": "system", "content": "Be brief."}, {"role": "user", "content": "Say hello in five words."}])["content"])

**Expected output:** a short greeting. An authentication or connection error means a wrong key, or (Ollama) that the server is not running or the model is not pulled.

## Exercise 1: Rebuild the Lab 3 index (recap)
**Why we do this:** the agent's first tool searches documents, so we need the same ChromaDB index as Lab 3. This is a shortened version; if you finished Lab 3 you have seen every line.

**What the code does:** downloads the SEBI memo, reads each page, cuts it into overlapping chunks, and stores them in Chroma with the document name and page as metadata. We also keep the raw page text in `PAGES`, because the third tool needs whole pages.

Put any extra PDFs (for example the RBI Master Direction) in the `data/` folder before running; they are loaded too.

In [ ]:
import re, urllib.request
import chromadb
from pypdf import PdfReader

os.makedirs("data", exist_ok=True)
SEBI_URL = "https://www.sebi.gov.in/sebi_data/meetingfiles/dec-2024/1735042007618_1.pdf"
if not os.path.exists("data/sebi_ai_memo.pdf"):
    req = urllib.request.Request(SEBI_URL, headers={"User-Agent": "Mozilla/5.0"})
    open("data/sebi_ai_memo.pdf", "wb").write(urllib.request.urlopen(req).read())

PAGES = {}   # {file name: {page number: text}}
for f in os.listdir("data"):
    if f.lower().endswith(".pdf"):
        PAGES[f] = {i + 1: re.sub(r"\s+", " ", p.extract_text() or "").strip()
                    for i, p in enumerate(PdfReader(f"data/{f}").pages)}

def chunk(text, size=800, overlap=150):
    return [text[s:s + size] for s in range(0, max(len(text), 1), size - overlap) if text[s:s + size].strip()]

col = chromadb.Client().get_or_create_collection("lab4")
ids, docs, metas = [], [], []
for f, pages in PAGES.items():
    for pg, text in pages.items():
        for j, piece in enumerate(chunk(text)):
            ids.append(f"{f}-{pg}-{j}"); docs.append(piece); metas.append({"doc": f, "page": pg})
col.add(ids=ids, documents=docs, metadatas=metas)
print("documents:", list(PAGES), "| chunks stored:", col.count())

**Expected output:** the document names and a chunk count. If `PAGES` is empty, no PDF was found in `data/`.

## Exercise 2: Define three tools
**Why we do this:** a tool is just a normal Python function plus a short description the model can read. The model never runs code itself: it only *asks* for a tool by name with arguments, and our program runs it. The description is how the model decides which tool fits (Module 4, section 2).

**The three tools:**
- `search_documents(query, k)` looks up passages in the Chroma index (retrieval).
- `calculate_ratio(numerator, denominator)` does arithmetic in plain Python. Models are unreliable at arithmetic, so we never let them do it in their head.
- `summarise_section(doc, page)` fetches one whole page and asks the model to summarise it.

**Key terms:** *Tool schema* = the JSON description (name, purpose, parameters) sent to the model. *Deterministic* = the same input always gives the same output, like `calculate_ratio`.

**Your task:** write `run_tool`, which looks up the tool by name, runs it with the model's arguments, and returns the result as text. If anything fails it must return an error message instead of crashing, so the model can recover. Hint: use a dictionary mapping names to functions, and `try/except`.

In [ ]:
def search_documents(query: str, k: int = 3) -> str:
    r = col.query(query_texts=[query], n_results=int(k))
    return "\n\n".join(f"[{m['doc']} p.{m['page']}] {t}" for t, m in zip(r["documents"][0], r["metadatas"][0]))

def calculate_ratio(numerator: float, denominator: float, as_percent: bool = True) -> str:
    if float(denominator) == 0:
        return "Error: denominator is zero."
    value = float(numerator) / float(denominator)
    return f"{value * 100:.2f}%" if as_percent else f"{value:.4f}"

def summarise_section(doc: str, page: int) -> str:
    text = PAGES.get(doc, {}).get(int(page))
    if not text:
        return f"Error: no page {page} in '{doc}'. Documents: {list(PAGES)}"
    reply = chat([{"role": "system", "content": "Summarise in 3 bullet points. Use only the given text."},
                  {"role": "user", "content": text}])
    return reply["content"]

TOOLS = [
    {"type": "function", "function": {
        "name": "search_documents",
        "description": "Search the loaded regulatory documents and return the most relevant passages with document name and page.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string", "description": "What to look for"},
            "k": {"type": "integer", "description": "Number of passages (default 3)"}}, "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "calculate_ratio",
        "description": "Divide numerator by denominator exactly. Use for ANY ratio or percentage; never calculate by hand.",
        "parameters": {"type": "object", "properties": {
            "numerator": {"type": "number"}, "denominator": {"type": "number"},
            "as_percent": {"type": "boolean", "description": "Return as a percentage (default true)"}},
            "required": ["numerator", "denominator"]}}},
    {"type": "function", "function": {
        "name": "summarise_section",
        "description": "Summarise one whole page of a loaded document. Loaded documents: " + ", ".join(PAGES),
        "parameters": {"type": "object", "properties": {
            "doc": {"type": "string", "description": "Exact file name"},
            "page": {"type": "integer", "description": "Page number, starting at 1"}}, "required": ["doc", "page"]}}},
]

def run_tool(name: str, arguments: str) -> str:
    # YOUR CODE HERE:
    # 1. Map tool names to the functions above.
    # 2. Parse `arguments` (a JSON string) with json.loads.
    # 3. Call the function with those arguments and return the result as a string.
    # 4. If the name is unknown or anything raises, return "Error: ..." instead of crashing.
    raise NotImplementedError

<details><summary>Solution (try first)</summary>

```python
def run_tool(name, arguments):
    registry = {"search_documents": search_documents,
                "calculate_ratio": calculate_ratio,
                "summarise_section": summarise_section}
    if name not in registry:
        return f"Error: unknown tool '{name}'."
    try:
        return str(registry[name](**json.loads(arguments or "{}")))
    except Exception as e:
        return f"Error: {e}"
```
</details>

In [ ]:
print(run_tool("calculate_ratio", '{"numerator": 1350, "denominator": 10000}'))
print(run_tool("calculate_ratio", '{"numerator": 1, "denominator": 0}'))
print(run_tool("no_such_tool", "{}"))

**Expected output:** `13.50%`, then an error about a zero denominator, then an error about an unknown tool. The errors are returned as text, not raised.

**What to notice:** returning the error as text lets the model read it and try something else, instead of the whole agent crashing.

**Finance link:** `calculate_ratio` keeps every number reproducible. An auditor can re-run the exact calculation, which a model's mental arithmetic does not allow.

## Exercise 3: The ReAct loop
**Why we do this:** ReAct means *Reason, then Act, then Observe, repeat* (Module 4, section 4). The model reasons about what it needs, requests a tool (act), we run it and show the result (observe), and the model decides whether it has enough to answer or needs another tool.

**What the code does:** keeps a growing `messages` list. Each round it asks the model what to do. If the reply contains tool calls, we run each one and append the result as a `tool` message, then loop. If the reply has no tool calls, that reply is the final answer. `max_steps` stops the loop if the model never finishes.

**Key terms:** *Step limit* = a hard cap on rounds, which prevents infinite loops and runaway cost. *Trace* = the printed record of each tool call and result.

**Why the cap matters:** each round is one paid or rate-limited model call.

In [ ]:
SYSTEM = (
    "You are a financial regulation research assistant. Use the tools to ground every factual claim. "
    "Use calculate_ratio for any arithmetic. Cite sources as [document p.N]. "
    "If the documents do not contain the answer, say so plainly rather than guessing."
)

def run_agent(question: str, max_steps: int = 6, verbose: bool = True) -> str:
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        msg = chat(messages, TOOLS)
        messages.append(msg)
        calls = msg.get("tool_calls")
        if not calls:
            return msg["content"]
        for call in calls:
            name, args = call["function"]["name"], call["function"]["arguments"]
            result = run_tool(name, args)
            if verbose:
                print(f"step {step}: {name}({args})\n   -> {result[:200]}{'...' if len(result) > 200 else ''}")
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})
    return "Stopped: step limit reached without a final answer."

print(run_agent("According to the SEBI memo, who is responsible for the output of AI tools?"))

**Expected output:** a trace showing a `search_documents` call, then a final answer citing the SEBI memo. The exact query the model writes will vary.

**What to notice:** the model chose to search without being told to. Now ask: *"A bank has total capital of 1,350 and risk-weighted assets of 10,000. What is its capital adequacy ratio?"* You should see `calculate_ratio` instead, and no document search.

**If it fails:** a model that answers without calling any tool, or writes the call as plain text, does not support tool calling well. Try a larger model or another provider.

## Exercise 4: Human-in-the-loop checkpoint
**Why we do this:** some answers should not go out without a person checking them. In finance that includes anything that sounds like a compliance conclusion, an approval, or a recommendation (Module 4, section 8, "autonomy and control").

**What the code does:** `is_high_stakes` flags a question using keywords. `run_with_approval` runs the agent, then, for flagged questions, shows the draft and asks a human to approve before returning it. A rejected answer is withheld.

**Your task:** complete `run_with_approval`. Hint: call `run_agent`, test `is_high_stakes`, ask with `input()`, and return the draft only if the reply starts with "y".

**Important:** a keyword list is a crude screen. It will miss some high-stakes questions and flag harmless ones. Real systems use risk tiers, classifiers and policy review. Treat this as the simplest possible version of the idea.

In [ ]:
HIGH_STAKES = ["approve", "reject", "loan", "recommend", "complian", "minimum", "breach", "penalt"]

def is_high_stakes(question: str) -> bool:
    q = question.lower()
    return any(word in q for word in HIGH_STAKES)

AUTO_APPROVE = False   # set True only for unattended test runs

def run_with_approval(question: str) -> str:
    # YOUR CODE HERE:
    # 1. draft = run_agent(question)
    # 2. if the question is not high-stakes, return draft
    # 3. otherwise print the draft, ask the human to approve, return draft if they type y,
    #    else return "Answer withheld: reviewer did not approve."
    # (honour AUTO_APPROVE: if True, skip input() and approve)
    raise NotImplementedError

<details><summary>Solution (try first)</summary>

```python
def run_with_approval(question):
    draft = run_agent(question)
    if not is_high_stakes(question):
        return draft
    print("\n--- HIGH-STAKES: DRAFT FOR REVIEW ---\n" + draft + "\n-------------------------------------")
    ok = True if AUTO_APPROVE else input("Approve this answer? (y/n): ").strip().lower().startswith("y")
    return draft if ok else "Answer withheld: reviewer did not approve."
```
</details>

## Exercise 5: Test the agent
**Why we do this:** an agent can fail in new ways: picking the wrong tool, looping, or answering without evidence. Run it on three different tasks and read the trace each time.

**Test questions:**
1. A document question, which should call `search_documents`.
2. A calculation with a compliance angle, which should call `calculate_ratio` and then pause for approval. The 9% figure is supplied in the question; verify the real minimum against the RBI's Basel III capital regulations before relying on it.
3. A summary request, which should call `summarise_section`.

In [ ]:
tests = [
    "According to the SEBI memo, who is responsible for the output of AI tools?",
    "A bank has total capital of 1,350 and risk-weighted assets of 10,000. What is its capital adequacy ratio, and does it meet a 9% minimum?",
    "Summarise page 3 of the SEBI memo (file sebi_ai_memo.pdf).",
]
for q in tests:
    print("\nQ:", q)
    print("A:", run_with_approval(q))

**Expected output:** question 1 uses search, question 2 uses `calculate_ratio` (13.50%) and then asks for your approval, question 3 uses `summarise_section`. If a different tool is chosen, read the tool descriptions: they are what the model relies on.

**What to notice:** change a tool's `description` to something vague and re-run. Tool selection usually gets worse, which shows how much the description matters.

## Stretch goals
1. Add a fourth tool, for example `list_documents()`, and see whether the agent uses it.
2. Add the RBI Master Direction and ask a question that needs both documents.
3. Count tool calls and rough token use per question to estimate cost.

## Common errors
- **The model never calls a tool (Ollama or small models):** use a model tagged "tools" in the Ollama library, or switch provider.
- **400 error mentioning tools or thought signatures (Gemini):** use `gemini-2.5-flash`; newer Gemini versions need extra fields passed back that this notebook does not handle.
- **429 or rate limit (Groq, Gemini, OpenRouter free):** wait, or lower `max_steps`. Each round is one request.
- **`NotImplementedError`:** you have not completed `run_tool` or `run_with_approval` yet.